# SEPA Precios — Exploración de Productos y Canasta Representativa

**Objetivo:** Explorar los datos del SEPA de **abril 2026** para identificar qué productos tienen alta cobertura a nivel nacional (cadenas y regiones) y construir una **canasta representativa para una familia tipo de 4 integrantes**.

**Fuentes de datos:**
- **Datos SEPA** (ZIPs semestrales): descarga directa desde SharePoint — no requiere Google Drive
- **Maestro de Productos** y **Maestro de Sucursales**: descarga automática desde el repositorio GitHub

**Estructura del notebook:**
1. Configuración y acceso a datos
2. Carga de datos SEPA — Abril 2026
3. Carga de maestros
4. Enriquecimiento y exploración
5. Análisis de cobertura
6. Construcción de la canasta
7. Exportación de resultados

---
> **Nota sobre precios:** Los valores en SEPA están en centavos (ej: `1699999` = `$16.999,99`). El notebook divide por 100. Verificar con la celda de validación en la Sección 2.

## 1. Configuración y acceso a datos

In [ ]:
!pip install openpyxl tqdm -q

import zipfile, gzip, io, os, base64, warnings
import requests
from pathlib import Path
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Librerías cargadas correctamente')

In [ ]:
# ===========================================================
# CONFIGURACIÓN — Solo tocar esta celda
# ===========================================================

# --- Fuente de los ZIPs SEPA ---
# 'sharepoint' : descarga automática desde el folder compartido (recomendado)
# 'gdrive'     : lee desde Google Drive (requiere montar Drive en la celda siguiente)
FUENTE_DATOS = 'sharepoint'

# URL del folder SharePoint (configurado como "Cualquier persona", sin contraseña)
SHAREPOINT_URL = 'https://uadeeduar-my.sharepoint.com/:f:/g/personal/sriverti_uade_edu_ar/IgDRgb-IGkb3S7EgZanqlV3CAbL86_6gx3jH3Ib8FMaBJjw?e=fU8Wax'

# Nombre del ZIP a descargar desde SharePoint
SEPA_ZIP_NAME = '2026A.zip'

# [Solo si FUENTE_DATOS='gdrive'] Ruta al ZIP en Google Drive
GDRIVE_ZIP_PATH = '/content/drive/MyDrive/SEPA/2026A.zip'

# Archivos de abril 2026 dentro del ZIP
ABRIL_PARTE1 = '042026_pais_parte1COMPLETO.csv.gz'
ABRIL_PARTE2 = '042026_pais_parte2COMPLETO.csv.gz'

# Directorio de trabajo local en Colab (persiste durante la sesión)
WORK_DIR = Path('/tmp/sepa_data')
OUT_DIR  = Path('/tmp/sepa_output/canasta_abril2026')
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ===========================================================
# PARÁMETROS DE COBERTURA
# El SEPA de abril 2026 tiene 6 cadenas y 6 regiones.
# ===========================================================
MIN_CADENAS    = 3     # >= 50% de las cadenas
MIN_REGIONES   = 4     # >= 67% de las regiones
MIN_SUCURSALES = 50    # mínimo de sucursales que reportan
MIN_PCT_DIAS   = 0.50  # precio disponible al menos el 50% de los días

print('Configuración cargada')
print(f'  Fuente de datos: {FUENTE_DATOS}')
print(f'  ZIP a procesar:  {SEPA_ZIP_NAME}')
print(f'  Archivos:        {ABRIL_PARTE1} | {ABRIL_PARTE2}')
print(f'  Umbrales:        {MIN_CADENAS} cadenas | {MIN_REGIONES} regiones | {MIN_SUCURSALES} sucursales | {MIN_PCT_DIAS*100:.0f}% días')

In [ ]:
# ===========================================================
# ACCESO A DATOS SEPA
# ===========================================================

def _sharepoint_token(folder_url: str) -> str:
    """Convierte una URL de SharePoint en un token para la Graph API."""
    encoded = base64.urlsafe_b64encode(folder_url.encode()).rstrip(b'=').decode()
    return f'u!{encoded}'


def listar_archivos_sharepoint(folder_url: str) -> list:
    """Lista los archivos del folder compartido vía Microsoft Graph API."""
    token = _sharepoint_token(folder_url)
    api_url = f'https://graph.microsoft.com/v1.0/shares/{token}/root/children'
    resp = requests.get(api_url, headers={'Accept': 'application/json'}, timeout=30)

    if resp.status_code == 200:
        return resp.json().get('value', [])

    raise RuntimeError(
        f'Graph API respondió {resp.status_code}.\n'
        f'Detalle: {resp.text[:400]}\n\n'
        f'Posibles causas:\n'
        f'  1. El tenant de UADE bloqueó el acceso anónimo a Graph API.\n'
        f'  2. La URL de SharePoint cambió.\n'
        f'Solución: cambiá FUENTE_DATOS = "gdrive" y montá Google Drive.'
    )


def descargar_de_sharepoint(folder_url: str, filename: str, dest_dir: Path) -> Path:
    """
    Descarga un archivo del folder SharePoint al directorio local.
    Usa cache: si el archivo ya existe no vuelve a descargarlo.
    """
    dest_path = dest_dir / filename
    if dest_path.exists():
        print(f'  Usando cache local: {dest_path}')
        return dest_path

    archivos = listar_archivos_sharepoint(folder_url)
    info = next((f for f in archivos if f['name'] == filename), None)

    if info is None:
        disponibles = [f['name'] for f in archivos]
        raise FileNotFoundError(
            f'No se encontró "{filename}" en el folder.\n'
            f'Archivos disponibles: {disponibles}'
        )

    download_url = info.get('@microsoft.graph.downloadUrl')
    size_mb = info.get('size', 0) / 1024 / 1024
    print(f'  Descargando {filename} ({size_mb:.0f} MB) ...')

    with requests.get(download_url, stream=True, timeout=600) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(dest_path, 'wb') as f, tqdm(
            total=total, unit='B', unit_scale=True, unit_divisor=1024,
            desc=filename, leave=True
        ) as bar:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
                bar.update(len(chunk))

    print(f'  Guardado en: {dest_path}')
    return dest_path


# --- Obtener la ruta al ZIP según la fuente elegida ---
if FUENTE_DATOS == 'sharepoint':
    print('Accediendo al folder de SharePoint...')
    try:
        archivos_sp = listar_archivos_sharepoint(SHAREPOINT_URL)
        print(f'  Archivos encontrados en el folder ({len(archivos_sp)}):')
        for a in archivos_sp:
            print(f'    {a["name"]}  ({a.get("size",0)/1024/1024:.0f} MB)')
        print()
        ZIP_PATH = descargar_de_sharepoint(SHAREPOINT_URL, SEPA_ZIP_NAME, WORK_DIR)
    except Exception as e:
        print(f'ERROR al acceder a SharePoint:\n{e}')
        print('\nCambiá FUENTE_DATOS = "gdrive" en la celda de Configuración y volvé a ejecutar.')
        raise

elif FUENTE_DATOS == 'gdrive':
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP_PATH = Path(GDRIVE_ZIP_PATH)
    assert ZIP_PATH.exists(), f'No se encontró el ZIP en: {ZIP_PATH}'
    print(f'ZIP cargado desde Google Drive: {ZIP_PATH}')

else:
    raise ValueError(f'FUENTE_DATOS debe ser "sharepoint" o "gdrive". Valor actual: {FUENTE_DATOS}')

print(f'\nZIP listo: {ZIP_PATH}')

In [ ]:
# ===========================================================
# MAESTROS — Descarga automática desde GitHub (repo público)
# No requiere Google Drive ni SharePoint
# ===========================================================

GITHUB_RAW = 'https://github.com/santiagoriverti/precios_minoristas_supermercados/raw/main/data/'

MAESTROS = {
    'Maestro de Productos Interno.xlsx': GITHUB_RAW + 'Maestro%20de%20Productos%20Interno.xlsx',
    'maestro_sucursales_completo.xlsx':  GITHUB_RAW + 'maestro_sucursales_completo.xlsx',
}

def descargar_maestro(nombre: str, url: str, dest_dir: Path) -> Path:
    dest = dest_dir / nombre
    if dest.exists():
        print(f'  Cache: {nombre}')
        return dest
    print(f'  Descargando {nombre} ...')
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    print(f'  Guardado ({len(resp.content)/1024:.0f} KB)')
    return dest

print('Descargando maestros desde GitHub...')
MAESTRO_PRODUCTOS_PATH = descargar_maestro('Maestro de Productos Interno.xlsx', MAESTROS['Maestro de Productos Interno.xlsx'], WORK_DIR)
MAESTRO_SUCURSALES_PATH = descargar_maestro('maestro_sucursales_completo.xlsx', MAESTROS['maestro_sucursales_completo.xlsx'], WORK_DIR)
print('Maestros listos')

## 2. Carga de datos SEPA — Abril 2026

Los archivos SEPA tienen el formato `MMAAAA_pais_parteN_COMPLETO.csv.gz` dentro de un ZIP semestral:
- **Parte 1:** días 1 al 15 del mes (15 columnas de precio)
- **Parte 2:** días 16 al 30/31 del mes (15 columnas de precio)
- **Precios:** entero en centavos, `NA` si la sucursal no reportó ese día

In [ ]:
import gc

def cargar_sepa(zip_path: Path, filename: str) -> pd.DataFrame:
    """
    Lee un .csv.gz desde dentro de un .zip.
    Devuelve DataFrame reducido: IDs + precio_promedio + dias_con_precio.
    Precios en pesos (divididos por 100 desde centavos).
    """
    print(f'  Leyendo {filename} ...')

    with zipfile.ZipFile(zip_path, 'r') as z:
        with z.open(filename) as zf:
            buf = io.BytesIO(zf.read())

    with gzip.open(buf, 'rt', encoding='utf-8') as g:
        df = pd.read_csv(
            g,
            dtype={
                'id_comercio': 'str', 'id_bandera': 'str',
                'id_sucursal': 'str', 'id_producto': 'str',
                'sucursales_provincia': 'str'
            },
            low_memory=False
        )

    price_cols = [c for c in df.columns if c.startswith('precio_')]
    n_dias = len(price_cols)

    df[price_cols] = df[price_cols].replace('NA', np.nan).astype(float) / 100

    df['precio_promedio']  = df[price_cols].mean(axis=1)
    df['dias_con_precio']  = df[price_cols].notna().sum(axis=1)
    df['total_dias_parte'] = n_dias

    print(f'    -> {len(df):,} filas | {df["id_producto"].nunique():,} productos únicos | {n_dias} días')

    return df[['id_comercio', 'id_bandera', 'id_sucursal', 'sucursales_provincia',
               'id_producto', 'precio_promedio', 'dias_con_precio', 'total_dias_parte']]

In [ ]:
print('Cargando SEPA Abril 2026...')
print('-' * 55)

df_p1 = cargar_sepa(ZIP_PATH, ABRIL_PARTE1)
df_p2 = cargar_sepa(ZIP_PATH, ABRIL_PARTE2)

df_abril = pd.concat([df_p1, df_p2], ignore_index=True)
del df_p1, df_p2; gc.collect()

# Consolidar por (producto × sucursal)
df_suc = df_abril.groupby(
    ['id_producto', 'id_bandera', 'id_comercio', 'id_sucursal', 'sucursales_provincia'],
    as_index=False
).agg(
    precio_promedio = ('precio_promedio',  'mean'),
    dias_con_precio = ('dias_con_precio',  'sum'),
    total_dias      = ('total_dias_parte', 'sum')
)
df_suc['pct_dias'] = df_suc['dias_con_precio'] / df_suc['total_dias']

del df_abril; gc.collect()

print('\nDatos consolidados (producto × sucursal):')
print(f'  Filas:             {len(df_suc):,}')
print(f'  Productos únicos:  {df_suc["id_producto"].nunique():,}')
print(f'  Cadenas:           {df_suc["id_bandera"].nunique()}')
print(f'  Provincias:        {df_suc["sucursales_provincia"].nunique()}')
print(f'  Sucursales:        {df_suc["id_sucursal"].nunique():,}')

In [ ]:
# Verificación de escala de precios
# Los 10 productos más reportados con sus precios medianos.
# Confirmar que /100 (centavos→pesos) da valores razonables.
print('=== Verificación de escala de precios ===')
print('Divisor aplicado: /100  (centavos → pesos)')
print()
top_obs = (
    df_suc.groupby('id_producto')
    .agg(n_sucursales=('id_sucursal','count'), precio_mediano=('precio_promedio','median'))
    .sort_values('n_sucursales', ascending=False)
    .head(10).reset_index()
)
print(top_obs.to_string(index=False))
print()
print('Si los precios parecen incorrectos, ajustar el divisor en cargar_sepa().')
print('Alternativas: /100 (centavos→pesos) | /1000 (milipesos→pesos)')

## 3. Carga de maestros

In [ ]:
print('Cargando Maestro de Productos...')
df_prod = pd.read_excel(MAESTRO_PRODUCTOS_PATH, dtype={'producto_sepa_id': str})
df_prod['id_producto'] = df_prod['producto_sepa_id'].str.strip()
df_prod = df_prod[df_prod['producto_blacklist'] == 0].copy()

df_prod_uniq = (
    df_prod[['id_producto', 'producto_descripcion', 'producto_marca',
             'rubro', 'categoria', 'subcategoria',
             'producto_cantidad_presentacion', 'producto_unidad_medida_presentac']]
    .drop_duplicates('id_producto')
)
print(f'  Productos únicos (sin blacklist): {len(df_prod_uniq):,}')
print(f'  Rubros ({df_prod_uniq["rubro"].nunique()}):')
print(df_prod_uniq['rubro'].value_counts().to_string())

In [ ]:
print('Cargando Maestro de Sucursales...')
df_suc_maest = pd.read_excel(
    MAESTRO_SUCURSALES_PATH,
    dtype={'id_comercio': str, 'id_bandera': str, 'id_sucursal': str}
)
df_suc_maest['REGION'] = df_suc_maest['REGION'].str.strip()

print(f'  Total sucursales:    {len(df_suc_maest):,}')
print(f'  Cadenas:             {df_suc_maest["id_bandera"].nunique()}')
print(f'  Regiones ({df_suc_maest["REGION"].nunique()}):')
print(df_suc_maest.groupby('REGION')['id_sucursal'].nunique().sort_values(ascending=False).to_string())
print('\nCadenas por sucursales:')
print(df_suc_maest.groupby('id_bandera')['id_sucursal'].nunique().sort_values(ascending=False).to_string())

## 4. Enriquecimiento y exploración

In [ ]:
suc_info = df_suc_maest[['id_comercio', 'id_bandera', 'id_sucursal',
                          'sucursales_nombre', 'PROVINCIA', 'REGION']].copy()

df_enr = df_suc.merge(suc_info, on=['id_comercio', 'id_bandera', 'id_sucursal'], how='left')
df_enr['REGION'] = df_enr['REGION'].str.strip()
df_enr = df_enr.merge(df_prod_uniq, on='id_producto', how='left')

print(f'Match maestro sucursales: {df_enr["REGION"].notna().mean()*100:.1f}% de filas')
print(f'Match maestro productos:  {df_enr["rubro"].notna().mean()*100:.1f}% de filas')
print(f'Productos sin clasificar: {df_enr[df_enr["rubro"].isna()]["id_producto"].nunique():,}')

In [ ]:
print('=== Distribución por cadena — Abril 2026 ===')
print(df_enr.groupby('id_bandera').agg(
    sucursales_activas   = ('id_sucursal',     'nunique'),
    productos_reportados = ('id_producto',     'nunique'),
    precio_mediano       = ('precio_promedio', 'median')
).sort_values('productos_reportados', ascending=False).to_string())

print('\n=== Distribución por región geográfica ===')
print(df_enr.groupby('REGION').agg(
    sucursales_activas   = ('id_sucursal', 'nunique'),
    productos_reportados = ('id_producto', 'nunique')
).sort_values('sucursales_activas', ascending=False).to_string())

In [ ]:
print('=== Top 20 productos más reportados ===')
top_prod = (
    df_enr.groupby(['id_producto', 'producto_descripcion', 'producto_marca', 'rubro', 'categoria'])
    .agg(
        n_sucursales = ('id_sucursal', 'count'),
        n_cadenas    = ('id_bandera',  'nunique'),
        n_regiones   = ('REGION',      lambda x: x.dropna().nunique()),
        precio_med   = ('precio_promedio', 'median')
    )
    .sort_values('n_sucursales', ascending=False).head(20).reset_index()
)
print(top_prod[['id_producto','producto_descripcion','producto_marca',
                'rubro','n_cadenas','n_regiones','n_sucursales','precio_med']].to_string(index=False))

## 5. Análisis de cobertura

Para cada producto se calculan:
- **n_cadenas / n_regiones / n_sucursales**: cobertura geográfica y de cadenas
- **pct_dias_promedio**: % de días de abril con precio reportado
- **score_cobertura**: índice compuesto = 50% cadenas + 50% regiones, ponderado por continuidad

In [ ]:
total_cadenas  = df_enr['id_bandera'].nunique()
total_regiones = df_enr['REGION'].dropna().nunique()
print(f'Cadenas activas: {total_cadenas} | Regiones activas: {total_regiones}')

df_cob = df_enr.groupby('id_producto').agg(
    n_cadenas         = ('id_bandera',          'nunique'),
    n_regiones        = ('REGION',              lambda x: x.dropna().nunique()),
    n_sucursales      = ('id_sucursal',          'count'),
    pct_dias_promedio = ('pct_dias',             'mean'),
    precio_mediano    = ('precio_promedio',       'median'),
    precio_promedio   = ('precio_promedio',       'mean'),
    precio_p25        = ('precio_promedio',       lambda x: x.quantile(0.25)),
    precio_p75        = ('precio_promedio',       lambda x: x.quantile(0.75)),
    rubro             = ('rubro',                 'first'),
    categoria         = ('categoria',             'first'),
    subcategoria      = ('subcategoria',          'first'),
    descripcion       = ('producto_descripcion',  'first'),
    marca             = ('producto_marca',         'first'),
    presentacion      = ('producto_cantidad_presentacion', 'first'),
    unidad            = ('producto_unidad_medida_presentac', 'first')
).reset_index()

df_cob['pct_cadenas']     = df_cob['n_cadenas']  / total_cadenas
df_cob['pct_regiones']    = df_cob['n_regiones'] / total_regiones
df_cob['score_cobertura'] = (
    (df_cob['pct_cadenas'] * 0.5 + df_cob['pct_regiones'] * 0.5)
    * df_cob['pct_dias_promedio']
)

print(f'\nProductos con al menos 1 observación: {len(df_cob):,}')
print(df_cob[['n_cadenas','n_regiones','n_sucursales','pct_dias_promedio']].describe().round(2).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribución de Cobertura — SEPA Abril 2026', fontsize=14, fontweight='bold')

for ax, col, bins, color, umbral, label in [
    (axes[0,0], 'n_cadenas',         range(0, total_cadenas+2), 'steelblue',    MIN_CADENAS,        'Cadenas'),
    (axes[0,1], 'n_regiones',        range(0, total_regiones+2),'seagreen',     MIN_REGIONES,       'Regiones'),
    (axes[1,0], 'n_sucursales',       40,                       'darkorange',   MIN_SUCURSALES,     'Sucursales'),
    (axes[1,1], 'pct_dias_promedio',  25,                       'mediumpurple', MIN_PCT_DIAS,       '% días'),
]:
    data = df_cob[col].clip(upper=600) if col == 'n_sucursales' else df_cob[col]
    kwargs = {'align': 'left'} if isinstance(bins, range) else {}
    ax.hist(data, bins=bins, color=color, edgecolor='white', **kwargs)
    ax.axvline(umbral if col != 'n_cadenas' else umbral - 0.5,
               color='crimson', linestyle='--', linewidth=1.5, label=f'Umbral: {umbral}')
    ax.set_title(f'N° de {label.lower()} por producto')
    ax.set_xlabel(label)
    ax.legend()
    if col == 'pct_dias_promedio':
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    elif isinstance(bins, range):
        ax.xaxis.set_major_locator(mticker.MultipleLocator(1))

plt.tight_layout()
plt.savefig(OUT_DIR / '01_distribucion_cobertura.png', dpi=150, bbox_inches='tight')
plt.show()

n_todos = (
    (df_cob['n_cadenas']         >= MIN_CADENAS)    &
    (df_cob['n_regiones']        >= MIN_REGIONES)   &
    (df_cob['n_sucursales']      >= MIN_SUCURSALES) &
    (df_cob['pct_dias_promedio'] >= MIN_PCT_DIAS)
).sum()
print(f'Productos que superan TODOS los umbrales: {n_todos:,} / {len(df_cob):,}')

In [ ]:
candidatos = df_cob[
    (df_cob['n_cadenas']         >= MIN_CADENAS)    &
    (df_cob['n_regiones']        >= MIN_REGIONES)   &
    (df_cob['n_sucursales']      >= MIN_SUCURSALES) &
    (df_cob['pct_dias_promedio'] >= MIN_PCT_DIAS)
].copy()
cand_con_maestro = candidatos[candidatos['rubro'].notna()].copy()

print(f'Productos candidatos (todos los filtros): {len(candidatos):,}')
print(f'  Con clasificación en maestro:           {len(cand_con_maestro):,}')
print(f'  Sin clasificar:                         {len(candidatos) - len(cand_con_maestro):,}')
print('\nCandidatos por rubro:')
print(cand_con_maestro['rubro'].value_counts().to_string())

In [ ]:
# Heatmaps de cobertura: top 40 candidatos × cadenas y × regiones
top_ids = cand_con_maestro.sort_values('score_cobertura', ascending=False).head(40)['id_producto'].tolist()
df_heat = (
    df_enr[df_enr['id_producto'].isin(top_ids)]
    .merge(candidatos[['id_producto','descripcion']], on='id_producto', how='left')
)
df_heat['label'] = df_heat['descripcion'].str[:45].fillna(df_heat['id_producto'])

for pivot_col, cmap, fname, title_suffix in [
    ('id_bandera', 'YlGnBu', '02_heatmap_cadenas.png',  '× cadena'),
    ('REGION',     'RdYlGn', '03_heatmap_regiones.png', '× región')
]:
    pivot = (
        df_heat.dropna(subset=[pivot_col])
        .groupby(['label', pivot_col])['pct_dias'].mean()
        .unstack(fill_value=0)
    )
    fig, ax = plt.subplots(figsize=(10, 14))
    sns.heatmap(pivot, cmap=cmap, linewidths=0.4, linecolor='white', vmin=0, vmax=1,
                cbar_kws={'label': '% días con precio', 'shrink': 0.6}, ax=ax)
    ax.set_title(f'Cobertura — Top 40 candidatos {title_suffix}', fontsize=13, pad=12)
    ax.tick_params(axis='y', labelsize=8)
    plt.tight_layout()
    plt.savefig(OUT_DIR / fname, dpi=150, bbox_inches='tight')
    plt.show()

## 6. Construcción de la canasta representativa

La canasta sigue la estructura de la **Canasta Básica Alimentaria (CBA)** del INDEC para una familia tipo de 4 integrantes, adaptada a los rubros del SEPA.

Dentro de cada grupo se seleccionan los productos con mayor `score_cobertura`.

In [ ]:
GRUPOS_CANASTA = {
    'Cereales y derivados':      {'rubros': ['Almacén'],                      'kw': ['arroz','pasta','fideo','harina','galletita','cereal','pan'],                  'max': 8},
    'Lácteos':                   {'rubros': ['Frescos','Almacén'],             'kw': ['leche','yogur','queso','crema','postre'],                                     'max': 8},
    'Aceites y grasas':          {'rubros': ['Almacén'],                      'kw': ['aceite','manteca','margarina'],                                               'max': 4},
    'Azúcar, dulces y conservas':{'rubros': ['Almacén'],                      'kw': ['azúcar','azucar','mermelada','dulce','tomate','conserva','legumbre'],          'max': 6},
    'Carnes y fiambres':         {'rubros': ['Frescos','Almacén','Congelados'],'kw': ['fiambre','embutido','carne','salchicha','pollo','atún','atun'],               'max': 6},
    'Huevos':                    {'rubros': ['Frescos','Almacén'],             'kw': ['huevo'],                                                                     'max': 2},
    'Condimentos y aderezos':    {'rubros': ['Almacén'],                      'kw': ['salsa','condimento','vinagre','mayonesa','mostaza','ketchup','aderezo'],       'max': 5},
    'Bebidas no alcohólicas':    {'rubros': ['Bebidas'],                      'kw': ['agua','gaseosa','jugo','saborizada','infusión','infusion','te','café','cafe'], 'max': 7},
    'Bebidas alcohólicas':       {'rubros': ['Bebidas'],                      'kw': ['cerveza','vino','sidra','fernet','espirituosa'],                              'max': 4},
    'Limpieza del hogar':        {'rubros': ['Limpieza'],                     'kw': None,                                                                          'max': 7},
    'Higiene y cuidado personal':{'rubros': ['Perfumería'],                   'kw': None,                                                                          'max': 6},
}

def seleccionar_grupo(df, rubros, keywords, max_n):
    subset = df[df['rubro'].isin(rubros)].copy()
    if keywords and len(subset) > 0:
        mask = subset['categoria'].str.contains('|'.join(keywords), case=False, na=False)
        filtered = subset[mask]
        subset = filtered if len(filtered) >= 2 else subset
    return subset.sort_values('score_cobertura', ascending=False).head(max_n)

partes = []
for grupo, cfg in GRUPOS_CANASTA.items():
    sel = seleccionar_grupo(cand_con_maestro, cfg['rubros'], cfg['kw'], cfg['max'])
    sel = sel.copy()
    sel['grupo_canasta'] = grupo
    partes.append(sel)
    print(f'{grupo}: {len(sel)} productos')

df_canasta = (
    pd.concat(partes, ignore_index=True)
    .drop_duplicates(subset='id_producto', keep='first')
)
print(f'\nTotal productos en la canasta: {len(df_canasta)}')

In [ ]:
cols_show = ['descripcion','marca','presentacion','unidad',
             'n_cadenas','n_regiones','n_sucursales','pct_dias_promedio','precio_mediano']

print('=' * 110)
print('CANASTA REPRESENTATIVA — FAMILIA TIPO 4 INTEGRANTES — ABRIL 2026')
print('=' * 110)

for grupo in GRUPOS_CANASTA:
    gdf = df_canasta[df_canasta['grupo_canasta'] == grupo]
    if len(gdf) == 0:
        print(f'\n[{grupo}] — sin productos que cumplan los umbrales')
        continue
    print(f'\n{"─"*110}\n  {grupo.upper()}  ({len(gdf)} productos)\n{"─"*110}')
    print(gdf[cols_show].sort_values('n_cadenas', ascending=False).to_string(index=False))

In [ ]:
resumen = (
    df_canasta.groupby('grupo_canasta')
    .agg(n_productos=('id_producto','count'), cob_cadenas=('n_cadenas','mean'),
         cob_regiones=('n_regiones','mean'),  precio_mediano=('precio_mediano','median'))
    .reset_index().sort_values('cob_cadenas', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Resumen de la Canasta — Abril 2026', fontsize=13, fontweight='bold')

ax = axes[0]
x, w = range(len(resumen)), 0.35
ax.barh([i+w/2 for i in x], resumen['cob_cadenas'],  w, label='Cadenas prom.',  color='steelblue')
ax.barh([i-w/2 for i in x], resumen['cob_regiones'], w, label='Regiones prom.', color='seagreen')
ax.set_yticks(list(x)); ax.set_yticklabels(resumen['grupo_canasta'], fontsize=9)
ax.set_title('Cobertura promedio por grupo'); ax.legend()

ax = axes[1]
rs = resumen.sort_values('precio_mediano')
ax.barh(rs['grupo_canasta'], rs['precio_mediano'],
        color=plt.cm.RdYlGn(rs['precio_mediano'] / rs['precio_mediano'].max()))
ax.set_title('Precio mediano por grupo (pesos)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig(OUT_DIR / '04_resumen_canasta.png', dpi=150, bbox_inches='tight')
plt.show()

# Box plot de dispersión de precios
orden = df_canasta.groupby('grupo_canasta')['precio_mediano'].median().sort_values(ascending=False).index.tolist()
fig, ax = plt.subplots(figsize=(14, 6))
bp = ax.boxplot([df_canasta[df_canasta['grupo_canasta']==g]['precio_mediano'].dropna().values for g in orden],
                vert=False, patch_artist=True, medianprops=dict(color='black', linewidth=2))
for patch, c in zip(bp['boxes'], plt.cm.tab20.colors):
    patch.set_facecolor(c); patch.set_alpha(0.75)
ax.set_yticks(range(1, len(orden)+1)); ax.set_yticklabels(orden, fontsize=9)
ax.set_title('Dispersión de precios por grupo — Abril 2026', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig(OUT_DIR / '05_dispersion_precios.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Exportación de resultados

In [ ]:
cols_export = ['grupo_canasta','id_producto','descripcion','marca','presentacion','unidad',
               'rubro','categoria','subcategoria','n_cadenas','pct_cadenas','n_regiones',
               'pct_regiones','n_sucursales','pct_dias_promedio','precio_mediano',
               'precio_promedio','precio_p25','precio_p75','score_cobertura']

canasta_export = (
    df_canasta[cols_export]
    .sort_values(['grupo_canasta','score_cobertura'], ascending=[True, False])
)

out_csv = OUT_DIR / 'canasta_representativa_abril2026.csv'
canasta_export.to_csv(out_csv, index=False, encoding='utf-8-sig')

out_cob = OUT_DIR / 'cobertura_candidatos_abril2026.csv'
cand_con_maestro.sort_values('score_cobertura', ascending=False).to_csv(out_cob, index=False, encoding='utf-8-sig')

print(f'Canasta exportada:            {out_csv}')
print(f'Cobertura completa exportada: {out_cob}')
print()
print(canasta_export.groupby('grupo_canasta').agg(
    productos      = ('id_producto',   'count'),
    cadenas_prom   = ('n_cadenas',      'mean'),
    regiones_prom  = ('n_regiones',     'mean'),
    precio_mediano = ('precio_mediano', 'median')
).round(1).to_string())

In [ ]:
# Descargar los CSVs al equipo local desde Colab
from google.colab import files

print('Descargando resultados al equipo local...')
files.download(str(out_csv))
files.download(str(out_cob))
print('Listo. También podés guardar en Google Drive si lo tenés montado:')
print('  import shutil')
print('  shutil.copy(out_csv, "/content/drive/MyDrive/SEPA/output/canasta_abril2026.csv")')